## 第7章 异常和上下文管理器

### 1.异常

- 抛出异常：`raise 异常类型(异常信息)`。
- 自定义异常：所有异常最终继承自`BaseException`，但**自定义异常应继承`Exception`**。如：`class MyException(Exception): pass`。
    - 异常继承体系如下：
        ```text
        BaseException
        ├── SystemExit              # sys.exit() 抛出
        ├── KeyboardInterrupt       # Ctrl+C 中断
        ├── GeneratorExit           # 生成器关闭
        └── Exception               # 所有常规异常的基类 ⭐
            ├── ValueError
            ├── TypeError
            ├── KeyError
            ├── IndexError
            ├── ZeroDivisionError
            ├── AttributeError
            ├── StopIteration
            └── ... (自定义异常)
        ```
- 处理异常：使用`try-except-else-finally`语句捕获并处理异常。
    - try语句后至少必须有一个except语句或finally语句。
    - 当一个异常被触发时，只有一个except代码块会被执行。因此，要把具体的异常类放在前面，再把通用异常类放在最后。
    - 在except语句中可以重新触发异常，使用`raise ... from`表明当前异常是由另一个异常导致的。使用`from None`可以完全抑制原始异常。

    ```python
    try:
        # 可能引发异常的代码
    except SomeError as e:
        # 捕获特定异常，用as保存异常对象
    except (OtherError, AnotherError) as e:
        # 捕获多种异常
    except Exception as e:
        # 捕获所有常规异常（兜底）
    else:
        # 没有异常时执行
    finally:
        # 总是执行
    ```

- 异常使用建议：

    | 建议                         | 说明                                         |
    | ---------------------------- | -------------------------------------------- |
    | ✅ 捕获具体异常               | `except ValueError` 而非 `except Exception`  |
    | ✅ 用 `else` 放成功逻辑       | 区分"可能出错的代码"和"只有成功才执行的代码"      |
    | ✅ 用 `finally` 做清理        | 确保资源释放                                 |
    | ✅ 自定义异常继承 `Exception` | 不要继承 `BaseException`                     |
    | ✅ 用 `raise from` 保留因果链 | 方便调试                                     |
    | ❌ 避免裸 `except:`           | 至少 `except Exception:`                     |
    | ❌ 不要用异常代替正常流程       | 如用 `try/except` 代替 `if/else` 做简单判断    |
    | ❌ 不要吞掉异常               | `except: pass` 隐藏了问题 ⚠️                  |

### 2.上下文管理器

- 上下文管理器：`with`语句用于管理资源，确保在使用后正确释放。
- 自定义上下文管理器：实现`__enter__`和`__exit__`方法的类，作为上下文管理器使用。
    - `__enter__()`：在进入`with`语句体之前调用，返回值会赋给`as`子句中的变量。
    - `__exit__(self, exc_type, exc_val, exc_tb)`：在`with`语句结束时调用，参数为异常类型、异常值和异常跟踪信息。
        - 如果没有异常发生，异常类型为`None`，异常值为`None`，异常跟踪信息为`None`。
        - 如果有异常发生，异常类型、异常值和异常跟踪信息会传递给`__exit__`方法。
        - 方法返回`True`时会抑制异常，返回`False`则继续传播异常。

In [ ]:
class MyContextManager:
    def __init__(self):
        print("初始化上下文管理器:", id(self))

    # 进入上下文管理器之前调用，返回值会赋给`as`子句中的变量
    def __enter__(self):
        print("进入上下文管理器")
        return self

    # 退出上下文管理器时调用，参数为异常类型、异常值和异常跟踪信息
    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"{exc_type=}, {exc_val=}, {exc_tb=}")
        print("退出上下文管理器")
        return True

ctx = MyContextManager()
print("开始使用上下文管理器")
with ctx as cm:
    print("上下文管理器中执行的代码")
    raise ValueError("这是一个测试异常")
print("上下文管理器执行结束")

- 基于生成器的上下文管理器：使用`contextlib`模块中的`@contextmanager`装饰器，可以将生成器函数转换为上下文管理器。
    - `__enter__()`方法启动这个生成器并返回生成器产生的对象.
    - 如果在`with`语句中发生异常，`__exit__()`方法将把这个异常传递给生成器(使用生成器的throw方法)​。
    - 如果在`with`语句中没有发生异常，`__exit__()`方法就在生成器上调用`next()`方法。

- 两种上下文管理器模板

**(1)类版模板：**

```python
class MyManager:
    def __enter__(self):
        # 获取资源
        return resource

    def __exit__(self, exc_type, exc_val, exc_tb):
        # 释放资源
        # return True → 抑制异常
        # return False → 异常继续传播(默认)
        return False
```

**生成器版模板：**

```python
from contextlib import contextmanager

@contextmanager
def my_manager():
    # 获取资源(等价于__enter__)
    try:
        yield resource
    finally:
        # 释放资源(等价于__exit__)
        pass
```

- 底层机制：

| 步骤         | 发生什么                                        |
| ------------ | ----------------------------------------------- |
| 1            | `with` 调用生成器函数，启动生成器               |
| 2            | 执行到 `yield`，`yield` 的值赋给 `as` 变量      |
| 3            | 执行 `with` 块体                                |
| 4a（无异常） | `finally` 执行，生成器结束                      |
| 4b（有异常） | 异常通过 `throw()` 传入生成器，`finally` 仍执行 |

In [ ]:
from contextlib import contextmanager

@contextmanager
def my_context():
    print("进入上下文管理器")  # __enter__ 逻辑
    try:
        yield "SomeResource"  # yield 的值 = __enter__ 的返回值
    except Exception as e:
        print(f"{type(e)=}, {e=}, {e.__traceback__=}")  # __exit__ 逻辑
    finally:
        print("退出上下文管理器")  # __exit__ 逻辑

print("开始使用上下文管理器")
with my_context() as val:
    print("上下文管理器中执行的代码")
    raise ValueError("这是一个测试异常")
print("上下文管理器使用结束")

### 3.本章小结

**核心知识脉络**

```text
异常与上下文管理器
│
├── 异常 ⭐
│   ├── 什么是异常（程序崩溃的原因）
│   ├── raise —— 主动抛出异常
│   ├── 自定义异常（继承 Exception）
│   ├── try/except/else/finally
│   │   ├── try → 可能出错的代码
│   │   ├── except → 捕获特定异常（具体在前，通用在后）
│   │   ├── else → 无异常时执行
│   │   └── finally → 无论如何都执行（清理）
│   ├── raise from —— 异常链（保留因果）
│   ├── 异常用于流程控制
│   │   ├── 跳出嵌套循环
│   │   ├── StopIteration（迭代器协议）
│   │   ├── NotImplementedError（抽象方法）
│   │   └── EAFP 风格：先尝试，出了问题再处理 ⭐
│   └── 常见内置异常
│
└── 上下文管理器 ⭐
    ├── 动机：确保资源释放（文件、锁、连接）
    ├── with 语句 = 自动 try/finally
    ├── 类版：__enter__ + __exit__
    │   ├── __enter__ → 返回资源
    │   ├── __exit__ → 清理（返回 True 抑制异常）
    │   └── exc_type / exc_val / exc_tb
    └── 生成器版：@contextmanager + yield ⭐
        ├── yield 前 = setup（__enter__）
        ├── yield 后 = cleanup（__exit__）
        └── 只能 yield 一次 ⚠️
```

**关键警告与提示**

| 类型   | 内容                                                         |
| ------ | ------------------------------------------------------------ |
| ⚠️ 警告 | 未处理的异常导致程序**立即退出**，后续代码不执行             |
| ⚠️ 警告 | 避免**裸 `except:`**，至少用 `except Exception:`             |
| ⚠️ 警告 | 自定义异常**必须继承 `Exception`**，不要继承 `BaseException` |
| ⚠️ 警告 | `except` 子句中**具体异常在前，通用异常在后**                |
| ⚠️ 警告 | `__exit__` 返回 `True` 会**抑制异常**（吞掉错误），慎用      |
| ⚠️ 警告 | 生成器版上下文管理器**只能 yield 一次**                      |
| ⚠️ 警告 | `raise ... from None` 会**丢失调试信息**，谨慎使用           |
| ⚠️ 警告 | `except: pass` 会**吞掉异常**，是极差的实践                  |
| 💡 技巧 | EAFP > LBYL："先做再说"比"先检查再行动"更 Pythonic ⭐         |
| 💡 技巧 | `else` 子句放"只有成功才做"的逻辑，让意图更清晰              |
| 💡 技巧 | `finally` 子句放清理逻辑，确保资源释放                       |
| 💡 技巧 | `raise from e` 保留异常因果链，方便调试                      |
| 💡 技巧 | 简单清理逻辑用生成器版上下文管理器；复杂状态管理用类版           |